# fashion_clip 상품 임베딩 시각화 — 유사 상품 그리드 + 카테고리 클러스터 산점도

**스파이크/발표 자료용 노트북입니다. 프로덕션 코드가 아닙니다.**

`app/recommend/similar/`(유사 상품 추천, CH020)가 쓰는 것과 **같은 임베딩**
(`product_garment_embedding` 테이블, fashion-clip zero-shot)을 가지고 두 가지를 보여준다:

1. 쿼리 상품 → 코사인 유사도 top-3 추천 상품 그리드 (서로 다른 카테고리 2~3세트)
2. 전체(또는 표본) 임베딩을 PCA/t-SNE로 2차원에 투영한 산점도 — 카테고리별 색 + 쿼리는 ★

## 전제 조건 — 먼저 실행할 것

이 노트북은 **DB/BE에 직접 접속하지 않는다.** `scripts/export_similar_product_samples.py`가
미리 조회해 로컬에 떨궈 둔 파일만 읽는다 — 그래야 시각화(matplotlib/scikit-learn 필요)와
조회(DB/BE 접속, `requirements.txt`만 필요) 의존성이 섞이지 않는다.

```bash
# ai 컨테이너 안에서 (torch 불필요 — 임베딩은 이미 계산되어 있다)
docker exec vintib-ai python scripts/export_similar_product_samples.py
```

그 뒤 이 노트북을 실행하려면 `scikit-learn`/`matplotlib`이 필요하다 —
`pip install -r validation/requirements-test.txt`(이미 두 패키지 모두 포함되어 있다).

## ⚠️ 카테고리 값이 깨끗하지 않을 수 있다

`product_garment_embedding.category`는 과거에 BE `categoryId`(숫자 문자열)와 AI
`GarmentCategory`(6종 enum 문자열)가 같은 컬럼에 섞여 저장된 적이 있다
(`app/recommend/personalize/embedding_store.py` 주석 참고). 산점도의 카테고리 색이
이상하게 나오면(예: 같은 옷 종류인데 색이 다르게 나뉨) 이 히스토리 때문일 수 있다 —
1단계 스크립트 출력의 "카테고리별 건수"를 먼저 확인할 것.

## 0. 설정

In [ ]:
import json
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import font_manager
from PIL import Image

DATA_DIR = Path("./data")
RESULTS_DIR = Path("./results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def use_korean_font():
    """그래프 제목·범례의 한글이 두부(□□)로 깨지지 않게 한글 폰트를 잡는다.

    QC 스파이크들의 qc_shared.use_korean_font()와 같은 후보 목록이다 — 이 노트북은
    REC 도메인 전용이라 QUAL 쪽 모듈을 import하지 않고 짧으니 그대로 복사해 둔다.
    """
    available = {f.name for f in font_manager.fontManager.ttflist}
    for candidate in ("Malgun Gothic", "AppleGothic", "NanumGothic", "Pretendard", "Gulim"):
        if candidate in available:
            matplotlib.rcParams["font.family"] = candidate
            matplotlib.rcParams["axes.unicode_minus"] = False
            return candidate
    return None


print(f"matplotlib 한글 폰트: {use_korean_font()}")

query_sets_path = DATA_DIR / "query_sets.json"
embeddings_path = DATA_DIR / "embeddings_sample.npz"
if not query_sets_path.exists() or not embeddings_path.exists():
    raise FileNotFoundError(
        f"{DATA_DIR} 에 필요한 파일이 없습니다. "
        "scripts/export_similar_product_samples.py 를 먼저 실행하세요."
    )

query_sets = json.loads(query_sets_path.read_text(encoding="utf-8"))
npz = np.load(embeddings_path, allow_pickle=False)
embeddings, categories, product_ids = npz["embeddings"], npz["categories"], npz["product_ids"]

print(f"쿼리 세트 {len(query_sets)}개")
print(f"산점도용 임베딩 {embeddings.shape[0]}건, 차원 {embeddings.shape[1]}")
print("카테고리 분포:")
_labels, _counts = np.unique(categories, return_counts=True)
for _l, _c in sorted(zip(_labels, _counts), key=lambda x: -x[1]):
    print(f"  {_l!r:<20} {_c:>5}건")

In [ ]:
def save_fig(fig, filename):
    path = RESULTS_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches="tight")
    w_in, h_in = fig.get_size_inches()
    print(f"저장: {path}  (약 {w_in * 300:.0f}x{h_in * 300:.0f}px @300dpi)")
    return path


def load_product_image(product_id):
    """1단계가 내려받아 둔 이미지를 연다. 없으면 None(호출부가 플레이스홀더를 그린다)."""
    path = DATA_DIR / "images" / f"{product_id}.jpg"
    if not path.exists():
        return None
    return Image.open(path).convert("RGB")


def show_image_or_placeholder(ax, product_id, ok):
    img = load_product_image(product_id) if ok else None
    if img is None:
        ax.text(
            0.5, 0.5, f"이미지 없음\n(product_id={product_id})",
            ha="center", va="center", transform=ax.transAxes, fontsize=10,
        )
        ax.set_facecolor("#f0f0f0")
    else:
        ax.imshow(img)
    ax.axis("off")

---

## 1. 쿼리 → top-3 추천 상품 그리드

쿼리 세트마다 별도 Figure — 1행 4열(쿼리 1장 + top-3 3장), 각 추천 상품 아래에 코사인
유사도 점수를 표시한다. 유사도는 1단계 스크립트가 `embedding_store.find_similar()`
(pgvector 코사인 거리, `app/recommend/similar/`가 서빙에 쓰는 것과 동일한 함수)로
이미 계산해 둔 값을 그대로 쓴다 — 이 노트북이 다시 계산하지 않는다.

In [ ]:
for qs in query_sets:
    n_cols = 1 + len(qs["neighbors"])
    fig, axes = plt.subplots(1, n_cols, figsize=(4.2 * n_cols, 5))

    show_image_or_placeholder(axes[0], qs["query_product_id"], qs["query_image_ok"])
    axes[0].set_title(
        f"쿼리\nproduct_id={qs['query_product_id']}\ncategory={qs['query_category']}",
        fontsize=10,
    )

    for col, neighbor in enumerate(qs["neighbors"], start=1):
        show_image_or_placeholder(axes[col], neighbor["product_id"], neighbor["image_ok"])
        axes[col].set_title(
            f"#{col} (유사도 {neighbor['score']:.3f})\n"
            f"product_id={neighbor['product_id']}\ncategory={neighbor['category']}",
            fontsize=10,
        )

    plt.suptitle(f"쿼리 category={qs['query_category']} — top-{len(qs['neighbors'])} 추천")
    plt.tight_layout()
    save_fig(fig, f"query_grid_{qs['query_product_id']}.png")
    plt.show()

---

## 2. 카테고리 클러스터 산점도 — PCA / t-SNE

같은 표본을 PCA(선형, 빠르고 결정적)와 t-SNE(비선형, 지역 구조를 더 잘 드러내지만 느리고
`random_state`에 따라 배치가 달라짐) 둘 다로 투영해서 나란히 비교할 수 있게 한다.
발표에는 클러스터가 더 선명하게 갈리는 쪽을 골라 쓰면 된다.

카테고리 종류가 너무 많으면(위 "⚠️ 카테고리 값이 깨끗하지 않을 수 있다" 참고) 범례가
안 읽히므로, 표본 안에서 건수가 많은 상위 14개 카테고리만 색을 따로 주고 나머지는
"기타"로 묶는다.

In [ ]:
_MAX_LEGEND_CATEGORIES = 14

_uniq, _counts = np.unique(categories, return_counts=True)
_top_categories = set(_uniq[np.argsort(-_counts)][:_MAX_LEGEND_CATEGORIES])
display_categories = np.array([c if c in _top_categories else "기타" for c in categories])
if len(_uniq) > _MAX_LEGEND_CATEGORIES:
    print(
        f"[알림] 카테고리가 {len(_uniq)}종이라 상위 {_MAX_LEGEND_CATEGORIES}종만 색을 구분하고 "
        f"나머지 {len(_uniq) - _MAX_LEGEND_CATEGORIES}종은 '기타'로 묶었습니다."
    )

query_product_id_set = {qs["query_product_id"] for qs in query_sets}
query_mask = np.isin(product_ids, list(query_product_id_set))
_missing_queries = query_product_id_set - set(product_ids[query_mask].tolist())
if _missing_queries:
    print(f"[경고] 산점도 표본에 없어 별표로 표시 못하는 쿼리: {sorted(_missing_queries)}")


def scatter_by_category(ax, xy, title):
    unique_cats = sorted(set(display_categories))
    cmap = plt.get_cmap("tab20")
    for i, cat in enumerate(unique_cats):
        mask = (display_categories == cat) & (~query_mask)
        ax.scatter(
            xy[mask, 0], xy[mask, 1], s=18, alpha=0.7, color=cmap(i % 20), label=str(cat)
        )
    # 쿼리는 항상 맨 위(zorder)에, 검은 테두리의 큰 별표로 강조한다.
    ax.scatter(
        xy[query_mask, 0], xy[query_mask, 1],
        s=420, marker="*", color="red", edgecolors="black", linewidths=1.2,
        zorder=5, label="쿼리 상품",
    )
    for pid, (x, y) in zip(product_ids[query_mask], xy[query_mask]):
        ax.annotate(f"product_id={pid}", (x, y), textcoords="offset points", offset=(8, 8), fontsize=9)
    ax.set_title(title)
    ax.legend(fontsize=7, loc="best", ncol=2, framealpha=0.9)

### 2-1. PCA

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
pca_xy = pca.fit_transform(embeddings)
_explained = pca.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(11, 9))
scatter_by_category(
    ax, pca_xy,
    f"PCA 2D — 상품 임베딩 (설명 분산 {sum(_explained):.1%}, PC1 {_explained[0]:.1%} / PC2 {_explained[1]:.1%})",
)
plt.tight_layout()
save_fig(fig, "scatter_pca.png")
plt.show()

### 2-2. t-SNE

In [ ]:
from sklearn.manifold import TSNE

_n = embeddings.shape[0]
if _n < 4:
    print(f"[건너뜀] 표본이 {_n}건뿐이라 t-SNE를 돌리기엔 너무 적습니다(최소 4건 권장).")
else:
    _perplexity = min(30, max(5, _n // 3))
    tsne = TSNE(n_components=2, random_state=42, perplexity=_perplexity, init="pca")
    tsne_xy = tsne.fit_transform(embeddings)

    fig, ax = plt.subplots(figsize=(11, 9))
    scatter_by_category(ax, tsne_xy, f"t-SNE 2D — 상품 임베딩 (perplexity={_perplexity})")
    plt.tight_layout()
    save_fig(fig, "scatter_tsne.png")
    plt.show()

---

## 결론 (실행 후 채울 것)

| # | 확인 항목 | 근거 셀 | 결과 |
| --- | --- | --- | --- |
| 1 | top-3 추천이 실제로 쿼리와 같은/유사한 카테고리·스타일인가 | §1 | |
| 2 | 카테고리별 클러스터가 산점도에서 실제로 뭉쳐 보이는가(PCA vs t-SNE 중 어느 쪽이 더 선명한가) | §2 | |
| 3 | 쿼리(★)가 자기 카테고리 클러스터 안에 있는가, 아니면 동떨어져 있는가 | §2 | |
| 4 | 카테고리 값 오염(§0 경고)이 해석에 영향을 줬는가 | §0 | |

### 한계

- 산점도는 전체가 아니라 표본(§0 카테고리 분포 참고)이다 — 표본이 작으면 클러스터 형태가
  실제 전체 분포를 과장/축소해 보여줄 수 있다.
- t-SNE는 축 자체(거리·방향)에 절대적 의미가 없다 — "가까이 모여 있다"만 읽고 "이 방향이
  무슨 의미다"까지는 해석하지 않는다.